The Goal of this notebook is to read all the Parquet files I parsed into, and determine just how much detectable error there is with a match threshold of 0.85. I should be able to find any "?" or any string that fails the validator

In [53]:
import calendar
import re

def is_valid_date(value: str) -> bool:
    """Validate '20YY/MM/DD' with YY in 10..26 and calendar-correct day."""
    if not isinstance(value, str):
        return False

    m = re.fullmatch(r"20(\d{2})/(\d{2})/(\d{2})", value)
    if not m:
        return False

    yy = int(m.group(1))
    mm = int(m.group(2))
    dd = int(m.group(3))

    if not (10 <= yy <= 26 and 1 <= mm <= 12):
        return False

    year = 2000 + yy
    max_day = calendar.monthrange(year, mm)[1]
    return 1 <= dd <= max_day


def is_valid_time(value: str) -> bool:
    """Validate 'HH:MM:SS' with HH 00..23, MM 00..59, SS 00..60."""
    if not isinstance(value, str):
        return False

    m = re.fullmatch(r"(\d{2}):(\d{2}):(\d{2})", value)
    if not m:
        return False

    hh = int(m.group(1))
    mm = int(m.group(2))
    ss = int(m.group(3))

    return 0 <= hh <= 23 and 0 <= mm <= 59 and 0 <= ss <= 60


def is_valid_filename(value: str) -> bool:
    """Validate filename consisting of exactly nine digits."""
    return isinstance(value, str) and re.fullmatch(r"\d{9}", value) is not None


In [37]:
import os
from glob import glob
import numpy as np
import pandas as pd
from dask import dataframe as dd
files = sorted(glob(os.path.join(os.getcwd(), '../..' , 'data', '*.parquet')))
# df = pd.read_parquet(files[0])
df = dd.read_parquet(files[0])
# np.char.find('?', df['date'].astype(str)) != -1
df['date']

Dask Series Structure:
npartitions=1
    string
       ...
Dask Name: getitem, 2 expressions
Expr=ReadParquetFSSpec(6ab0954)['date']

In [57]:
import os
from glob import glob
import numpy as np
import pandas as pd

files = sorted(glob(os.path.join(os.getcwd(), '../..' , 'data', '*.parquet')))



records = []
for file in files:
    df = pd.read_parquet(file)
    record = {'file': os.path.basename(file)}
    for col in df.columns:

        record[col + '_unknowns'] = df[col].str.contains('\\?').sum()
        record[col + '_schar'] = df[col].str.contains('s').sum()

        if col == 'date':
            invalids = ~df[col].apply(is_valid_date)
            record[col + '_invalid'] = invalids.sum()
        elif col == 'time':
            invalids = ~df[col].apply(is_valid_time)
            record[col + '_invalid'] = invalids.sum()
        elif col == 'filename':
            invalids = ~df[col].apply(is_valid_filename)
            record[col + '_invalid'] = invalids.sum()
        else:
            record[col + '_invalid'] = None
    records.append(record)


# mask = np.char.find(all_parsed, '?') != -1
# np.any(np.char.find(all_parsed, 's') != -1)
# np.count_nonzero(mask)
# pd.read_parquet(files[0])

pd.DataFrame.from_records(records)

,file,date_unknowns,date_schar,date_invalid,time_unknowns,time_schar,time_invalid,exposure_unknowns,exposure_schar,exposure_invalid,filename_unknowns,filename_schar,filename_invalid
0,2010-08.parquet,17,0,17,8,0,8,0,0,None,4,0,4
1,2010-09.parquet,0,0,0,0,0,0,0,0,None,0,0,0
2,2010-10.parquet,0,0,0,0,0,0,0,0,None,0,0,0
3,2010-11.parquet,0,0,0,0,0,0,0,0,None,3,0,3
4,2010-12.parquet,0,0,0,0,0,0,0,0,None,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
77,2020-08.parquet,15100,0,15100,15100,0,15100,0,0,None,15100,0,15100
78,2020-09.parquet,14656,0,14656,14655,0,14655,0,0,None,14656,0,14656
79,2020-10.parquet,15814,0,15814,15814,0,15814,0,0,None,15814,0,15814
80,2020-11.parquet,15386,0,15386,15386,0,15386,0,0,None,15386,0,15386
